# Bible GPT - A Simple Character-Level Language Model For Bible Text

In [3]:
import requests
from torch.testing._internal.distributed._tensor.common_dtensor import FeedForward

# Download the Bible text from Project Gutenberg
url = 'https://www.gutenberg.org/cache/epub/10/pg10.txt'
response = requests.get(url)
# Save the text to a file
with open('bible.txt', 'w', encoding='utf-8') as f:
    f.write(response.text)
with open('bible.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print(f'total characters in the text: {len(text)}')
print(text[:1000])  # Print the first 500 characters to check the content

total characters in the text: 4351850
﻿The Project Gutenberg eBook of The King James Version of the Bible
    
This ebook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this ebook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: The King James Version of the Bible

Release date: August 1, 1989 [eBook #10]
                Most recently updated: October 29, 2024

Language: English



*** START OF THE PROJECT GUTENBERG EBOOK THE KING JAMES VERSION OF THE BIBLE ***
The Old Testament of the King James Version of the Bible
The First Book of Moses: Called Genesis
The Second Book of Moses: Called Exodus
The Third Book of Moses: Called Leviticus
The Fourth Book of Mos

In [20]:
# TODO: Clean the text remove some title headers and footers
# We are to remove like '\ufeff' and 'tm'
text = text.replace('\ufeff', '').replace('™', '')

In [21]:
# TODO: Clean and get a very clean charset
chars = sorted(list(set(text)))
#chars = chars[:-2] # Remove the last character which is some character
vocab_size = len(chars)
print(f'vocab size: {vocab_size}')

vocab size: 86


In [22]:
chars

['\n',
 ' ',
 '!',
 '#',
 '$',
 '%',
 '(',
 ')',
 '*',
 ',',
 '-',
 '.',
 '/',
 '0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 ':',
 ';',
 '?',
 'A',
 'B',
 'C',
 'D',
 'E',
 'F',
 'G',
 'H',
 'I',
 'J',
 'K',
 'L',
 'M',
 'N',
 'O',
 'P',
 'Q',
 'R',
 'S',
 'T',
 'U',
 'V',
 'W',
 'X',
 'Y',
 'Z',
 '[',
 ']',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z',
 '—',
 '‘',
 '’',
 '“',
 '”',
 '•']

In [23]:
stoi = {ch: i for i, ch in enumerate(chars)}  # string to integer
itos = {i: ch for i, ch in enumerate(chars)}  # integer to string

encode = lambda s: [stoi[c] for c in s]  # encode a string to a list of integers
decode = lambda l: ''.join([itos[i] for i in l])  # decode a list of integers to a string

In [24]:
print(f'encode: {encode("In the beginning God created the heaven and the earth.")}')

encode: [34, 67, 1, 73, 61, 58, 1, 55, 58, 60, 62, 67, 67, 62, 67, 60, 1, 32, 68, 57, 1, 56, 71, 58, 54, 73, 58, 57, 1, 73, 61, 58, 1, 61, 58, 54, 75, 58, 67, 1, 54, 67, 57, 1, 73, 61, 58, 1, 58, 54, 71, 73, 61, 11]


## Creating a training and validation dataset

In [25]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)  # encode the text to a tensor of integers
n = int(0.9 * len(data))  # 90% for training, 10% for validation
train_data = data[:n]
val_data = data[n:]

block_size = 128

In [26]:
def get_batch(split, batch_size=32):
    data_source = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_source) - block_size, (batch_size,))  # random starting indices for each batch
    x = torch.stack([data_source[i:i+block_size] for i in ix])  # shape (batch_size, block_size)
    y = torch.stack([data_source[i+1:i+block_size+1] for i in ix])  # shape (batch_size, block_size)
    return x, y

# GPT Model Implementation

1. Input Ids (B,T)
2. Token Embedding (vocab_size, n_embed)
3. Positional Embedding (T, n_embed)
4. Add both embeddings together to get the input embeddings (B,T,n_embed)
5. Stack of Transformer Blocks (n_layer)
6. Final Linear Layer (n_embed, vocab_size)
7. Linear head (B,T,vocab_size)
8. Softmax to get the probabilities (B,T,vocab_size)


In [33]:
import torch.nn as nn
import torch.nn.functional as F
import torch

class GPTConfig:
    def __init__(self, vocab_size, block_size, n_embed=128, n_heads=4, n_layers=4, dropout=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_embed = n_embed
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.dropout = dropout

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embed)
        self.position_embedding = nn.Embedding(config.block_size, config.n_embed)

        self.blocks = nn.Sequential(*[
            TransformerBlock(config) for _ in range(config.n_layers)
        ])

        self.ln_f = nn.LayerNorm(config.n_embed)
        self.head = nn.Linear(config.n_embed, config.vocab_size)

        self.block_size = config.block_size

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # Check if input is too long
        assert T <= self.block_size

        # Embedding: tokens + positions
        tok_emb = self.token_embedding(idx)                 # (B, T, C)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))  # (T, C)
        x = tok_emb + pos_emb                               # (B, T, C)

        x = self.blocks(x)                                  # Transformer layers
        x = self.ln_f(x)                                     # Final LayerNorm
        logits = self.head(x)                               # (B, T, vocab_size)

        # If training: compute loss
        if targets is None:
            return logits
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
            return logits, loss

In [29]:
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embed)
        self.attn = MultiHeadAttention(config)
        self.ln2 = nn.LayerNorm(config.n_embed)
        self.mlp = FeedForward(config)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))  # Add & Norm
        x = x + self.mlp(self.ln2(x))  # Add & Norm
        return x

## Self Attention Mechanism

In [30]:
class AttentionHead(nn.Module):
    def __init__(self, head_size, config):
        super().__init__()
        self.key = nn.Linear(config.n_embed, head_size, bias=False)
        self.query = nn.Linear(config.n_embed, head_size, bias=False)
        self.value = nn.Linear(config.n_embed, head_size, bias=False)
        self.dropout = nn.Dropout(config.dropout)

        self.register_buffer("tril", torch.tril(torch.ones(config.block_size, config.block_size)))  # Lower triangular matrix for masking

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * (C ** -0.5)  # Scaled dot-product attention
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # Apply the mask
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)  # Apply dropout
        v = self.value(x)
        out = wei @ v  # (B, T, head_size)
        return out

## Multi-Head Attention

In [31]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embed % config.n_heads == 0, "n_embed must be divisible by n_heads"
        head_size = config.n_embed // config.n_heads
        self.heads = nn.ModuleList([
            AttentionHead(head_size, config) for _ in range(config.n_heads)
        ])
        self.proj = nn.Linear(config.n_embed, config.n_embed)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # Concatenate the outputs of all heads
        out = self.proj(out)  # Project back to n_embed dimension
        out = self.dropout(out)  # Apply dropout
        return out

## Feed Forward Network

In [32]:
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embed, 4 * config.n_embed),
            nn.GELU(),
            nn.Linear(4 * config.n_embed, config.n_embed),
            nn.Dropout(config.dropout)
        )
    def forward(self, x):
        return self.net(x)  # (B, T, n_embed)

# Training the Model

In [48]:
# Initialize the model
device = "mps" if torch.backends.mps.is_available() else "cpu"
device = "cpu"
config = GPTConfig(vocab_size=vocab_size, block_size=block_size, n_embed=128, n_heads=4, n_layers=4, dropout=0.1)
model = GPT(config)
# Optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10000)
loss_fn = nn.CrossEntropyLoss()

# Training loop
def train(model, get_batch, epochs=10000, batch_size=32):
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()

        xb, yb = get_batch('train', batch_size)
        xb, yb = xb.to(device), yb.to(device)

        logits = model(xb)
        loss = loss_fn(logits.view(-1, logits.size(-1)), yb.view(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        train_losses.append(loss.item())

        if epoch % 100 == 0:
            print(f"[Epoch {epoch:05d}] Train Loss: {loss.item():.4f}")

        # Validation
        if epoch % 500 == 0:
            model.eval()
            with torch.no_grad():
                val_xb, val_yb = get_batch('val', batch_size)
                val_xb, val_yb = val_xb.to(device), val_yb.to(device)

                val_logits = model(val_xb)
                val_loss = loss_fn(val_logits.view(-1, val_logits.size(-1)), val_yb.view(-1))
                val_losses.append(val_loss.item())

                print(f"Validation Loss @ Epoch {epoch}: {val_loss.item():.4f}")
            model.train()

    return train_losses, val_losses

In [49]:
train(model, get_batch, epochs=1000, batch_size=32)

[Epoch 00000] Train Loss: 4.6058
🔍 Validation Loss @ Epoch 0: 4.4815
[Epoch 00100] Train Loss: 2.6795
[Epoch 00200] Train Loss: 2.6364
[Epoch 00300] Train Loss: 2.5824
[Epoch 00400] Train Loss: 2.5873
[Epoch 00500] Train Loss: 2.4975
🔍 Validation Loss @ Epoch 500: 2.6324
[Epoch 00600] Train Loss: 2.5209
[Epoch 00700] Train Loss: 2.5152
[Epoch 00800] Train Loss: 2.5209
[Epoch 00900] Train Loss: 2.5178


([4.605844020843506,
  4.479134559631348,
  4.377433776855469,
  4.296091079711914,
  4.163498401641846,
  4.086144924163818,
  4.008152961730957,
  3.9122154712677,
  3.831451416015625,
  3.775843858718872,
  3.692594289779663,
  3.6114320755004883,
  3.517043113708496,
  3.489847421646118,
  3.4649245738983154,
  3.3867347240448,
  3.379657506942749,
  3.327554941177368,
  3.319545030593872,
  3.2744200229644775,
  3.242640256881714,
  3.199535369873047,
  3.1920411586761475,
  3.1908068656921387,
  3.1683223247528076,
  3.1446328163146973,
  3.118466377258301,
  3.1047699451446533,
  3.0818653106689453,
  3.0626580715179443,
  3.06758189201355,
  3.034864902496338,
  3.0435829162597656,
  3.0279431343078613,
  3.0559115409851074,
  3.0469884872436523,
  2.98006272315979,
  2.9886019229888916,
  2.959296464920044,
  2.995715856552124,
  2.919074773788452,
  2.905614137649536,
  2.92702054977417,
  2.912977695465088,
  2.935565233230591,
  2.8943095207214355,
  2.944197416305542,
  2.

In [50]:
# Generate text from the model
@torch.no_grad()
def generate(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits = model(idx_cond)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, next_id), dim=1)
    return idx

# Start with some character
context = torch.tensor([[stoi['G']]], dtype=torch.long)
out = generate(model, context, 200)
print(decode(out[0].tolist()))

Gonot h4Eraperes, yotyali ol, o souae sthito the thAcang uR6:1“loub; o ant
rras9thalw mrthe:17 thanety w tof 3i
iat O stod. the:heme  unae tominallbas, t gen
:e s yi ,:2 ar sanMorskhitsosaay, ayal5 th

